# Predictive Modeling (Regression) — Internship Task 3
**Python · Jupyter Notebook · Pandas · NumPy · Scikit-learn**

## Project objective
Build and evaluate a regression model using retail transaction data. This notebook predicts **line-level revenue** from quantity and unit price, demonstrates train/test splitting, feature selection, multicollinearity checks, model evaluation with RMSE/MAE/R², and prediction on new examples.

## Dataset
Download the UCI Online Retail dataset:
https://archive.ics.uci.edu/dataset/352/online+retail

Place `Online Retail.xlsx` beside this notebook, or update `FILE_PATH` below.

## Important modeling note
Revenue is calculated as `Quantity × UnitPrice`. Using Quantity and UnitPrice as predictors makes this a useful workflow demonstration, but it is also a near-deterministic business formula. Strong scores would not mean the model has discovered a general pattern. The optional model is intended for demonstrating the regression workflow. For a more meaningful forecasting project, predict a future month's revenue using only information available before that month.

## 1. Install packages

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn openpyxl

## 2. Import libraries and load data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.base import clone

sns.set_theme(style="whitegrid")

FILE_PATH = Path("Online Retail.xlsx")
if not FILE_PATH.exists():
    raise FileNotFoundError(
        "Place 'Online Retail.xlsx' beside this notebook or update FILE_PATH."
    )

df = pd.read_excel(FILE_PATH)
df.columns = df.columns.astype(str).str.strip()

print(f"Dataset size: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3. Prepare the modeling data

In [ ]:
# Convert fields to numeric/date types
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Create target: line-level revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Remove exact duplicates and rows without a usable target
model_df = df.drop_duplicates().copy()
model_df = model_df.dropna(subset=["Revenue", "Quantity", "UnitPrice"])

# Build date features without using the raw invoice date
model_df["InvoiceYear"] = model_df["InvoiceDate"].dt.year
model_df["InvoiceMonth"] = model_df["InvoiceDate"].dt.month
model_df["InvoiceHour"] = model_df["InvoiceDate"].dt.hour

# Keep a simple feature set for a reproducible baseline
feature_cols = ["Quantity", "UnitPrice", "InvoiceMonth", "InvoiceHour", "Country"]
target_col = "Revenue"

model_df = model_df.dropna(subset=["Revenue"])
display(model_df[feature_cols + [target_col]].head())
print("Modeling rows:", f"{len(model_df):,}")

## 4. Define features and target
- **Target (`y`)**: Revenue, the value the model will predict.
- **Features (`X`)**: Quantity, UnitPrice, month, hour, and country.
- We exclude `InvoiceNo`, `CustomerID`, and product description from this baseline to avoid an unnecessarily high-cardinality model and potential memorization.

In [ ]:
X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

numeric_features = ["Quantity", "UnitPrice", "InvoiceMonth", "InvoiceHour"]
categorical_features = ["Country"]

print("Feature rows:", X.shape[0])
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 5. Split training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Build preprocessing and regression pipeline
Numerical missing values are imputed with the median and scaled. Categorical missing values are filled with the most frequent value and one-hot encoded. The pipeline fits preprocessing only on training data, helping prevent data leakage.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression, k="all")),
    ("regressor", LinearRegression())
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("Model training completed.")

## 7. Evaluate model performance (RMSE, MAE, R²)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R-squared"],
    "Value": [mae, rmse, r2]
})
display(metrics)

print(f"MAE: {mae:,.4f}")
print(f"RMSE: {rmse:,.4f}")
print(f"R²: {r2:,.4f}")

**Metric interpretation**
- **MAE**: average absolute prediction error, in the same units as Revenue.
- **RMSE**: penalizes large errors more heavily than MAE.
- **R²**: compares the model with a baseline that predicts the test-set mean; it can be negative when predictions perform worse than that baseline.

Do not report assumed metrics. Use the values generated when you run the notebook.

## 8. Actual vs predicted plot

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.35)
min_val = min(float(y_test.min()), float(np.min(y_pred)))
max_val = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual revenue")
plt.ylabel("Predicted revenue")
plt.title("Actual vs Predicted Revenue")
plt.tight_layout()
plt.show()

## 9. Residual analysis

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Distribution of Residuals")
plt.xlabel("Residual (actual - predicted)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted revenue")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted Revenue")
plt.tight_layout()
plt.show()

## 10. Feature selection
SelectKBest with an F-test ranks individual transformed features by their univariate association with the target. This is a demonstration of feature selection, not proof of causality. It may select redundant features and should be combined with domain knowledge.

In [ ]:
fitted_preprocessor = model_pipeline.named_steps["preprocess"]
feature_names = fitted_preprocessor.get_feature_names_out()
selector = model_pipeline.named_steps["feature_selection"]
scores = selector.scores_

feature_scores = pd.DataFrame({
    "Feature": feature_names,
    "F_Score": scores
}).sort_values("F_Score", ascending=False)

display(feature_scores.head(20))

## 11. Check multicollinearity
Multicollinearity occurs when predictors are strongly related to each other. Check correlations between numeric predictors. One-hot encoded country indicators can also be related, so a full VIF analysis would need careful handling of the encoded design matrix.

The model target is Revenue = Quantity × UnitPrice, so very high predictive performance is expected from those inputs and is not evidence of a generalizable forecasting model.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    X_train[numeric_features].corr(),
    annot=True, fmt=".2f", cmap="coolwarm", center=0
)
plt.title("Correlation Between Numeric Predictors")
plt.tight_layout()
plt.show()

## 12. Make predictions for new examples

In [ ]:
new_data = pd.DataFrame({
    "Quantity": [2, 5, 10],
    "UnitPrice": [12.50, 7.99, 3.25],
    "InvoiceMonth": [11, 12, 1],
    "InvoiceHour": [10, 14, 9],
    "Country": ["United Kingdom", "France", "Germany"]
})

new_predictions = model_pipeline.predict(new_data)
new_data["Predicted_Revenue"] = new_predictions

display(new_data)

**Important:** These are illustrative predictions from the fitted model, not confirmed future sales. For future predictions, the input features must be available at prediction time and should match the real use case.

## 13. Save the model and outputs

In [ ]:
from pathlib import Path
import joblib

output_dir = Path("regression_output")
output_dir.mkdir(exist_ok=True)

metrics.to_csv(output_dir / "model_metrics.csv", index=False)
feature_scores.to_csv(output_dir / "feature_scores.csv", index=False)
new_data.to_csv(output_dir / "sample_predictions.csv", index=False)

joblib.dump(model_pipeline, output_dir / "linear_regression_pipeline.joblib")

print("Model and outputs saved to:", output_dir.resolve())

## 14. Conclusion template — fill in after running
- Dataset rows used for modeling: **[actual number]**
- Target variable: **Revenue**
- Features used: **Quantity, UnitPrice, InvoiceMonth, InvoiceHour, Country**
- MAE: **[actual value]**
- RMSE: **[actual value]**
- R²: **[actual value]**
- Multicollinearity observations: **[actual numeric feature correlations]**
- Selected high-scoring features: **[actual feature-score output]**

## Limitations and next step
This model predicts transaction-line revenue from quantity and unit price, which are components of the revenue formula. For a more meaningful business forecast, aggregate revenue by day or month and predict a future period using lagged revenue, calendar features, and only information available before the prediction period. Use a time-based train/test split for that forecasting task.